# Customer Churn Prediction | EDA, LightGBM & Business Insights

**Kaggle Playground Series S6E3 — Predict Customer Churn (2026)**

This notebook presents a reproducible workflow covering exploratory data analysis, a Logistic Regression baseline, LightGBM, validation metrics, risk segmentation, and **hypothetical** retention economics.

**Important:** The competition dataset is synthetic. Model associations are not causal effects; simulated retention profits are not realized business outcomes.

[GitHub project](https://github.com/tj-xin/customer-churn-analytics)

## 1. Load competition data

The input is attached from the Kaggle competition. We discover the input folder rather than assuming a fixed directory name.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

roots = list(Path('/kaggle/input').rglob('train.csv'))
assert roots, 'Attach the Predict Customer Churn competition dataset in Add Input.'
# Prefer a folder containing all three competition files.
candidates = [p.parent for p in roots if (p.parent / 'test.csv').exists() and (p.parent / 'sample_submission.csv').exists()]
assert len(candidates) == 1, f'Expected one matching competition input folder, found {candidates}'
data_dir = candidates[0]
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
sample_submission = pd.read_csv(data_dir / 'sample_submission.csv')
print('Data directory:', data_dir)
print('Train:', train.shape, '| Test:', test.shape, '| Submission:', sample_submission.shape)
display(train.head())

## 2. Data quality and exploratory analysis

In [ ]:
print('Churn distribution:')
display(train['Churn'].value_counts())
print('Churn rate:', f"{train['Churn'].eq('Yes').mean():.2%}")
print('Missing values:', int(train.isna().sum().sum()))
print('Duplicate rows:', int(train.duplicated().sum()))
print('Unique IDs:', train['id'].is_unique)
display(train[['tenure', 'MonthlyCharges', 'TotalCharges']].describe())

In [ ]:
contract_rates = train.assign(churn_flag=train['Churn'].eq('Yes')).groupby('Contract')['churn_flag'].mean().sort_values(ascending=False)
display((contract_rates * 100).round(2).rename('Churn rate (%)').to_frame())
ax = (contract_rates * 100).plot.bar(figsize=(7, 4), title='Observed churn rate by contract type')
ax.set_ylabel('Churn rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
eda = train.copy()
eda['TenureGroup'] = pd.cut(eda['tenure'], bins=[0, 12, 24, 48, 72], labels=['1-12', '13-24', '25-48', '49-72'])
weights = eda['TenureGroup'].value_counts(normalize=True).sort_index()
rates = eda.assign(churn_flag=eda['Churn'].eq('Yes')).groupby(['TenureGroup', 'Contract'], observed=True)['churn_flag'].mean().unstack('Contract')
standardized = rates.mul(weights, axis=0).sum(axis=0)
display(pd.DataFrame({'Observed churn (%)': contract_rates * 100, 'Tenure-standardized churn (%)': standardized * 100}).round(2))
print('Tenure–TotalCharges Pearson r:', round(train['tenure'].corr(train['TotalCharges']), 4))
eda['ChargeGroup'] = pd.qcut(eda['MonthlyCharges'], q=4, duplicates='drop')
display((eda.assign(churn_flag=eda['Churn'].eq('Yes')).groupby('ChargeGroup', observed=True)['churn_flag'].mean() * 100).round(2).rename('Churn rate (%)').to_frame())

**Interpretation:** Contract type is strongly associated with churn, including after tenure-group standardization. These are observational comparisons, not evidence that changing contracts would prevent churn.

## 3. Stratified validation split and Logistic Regression baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

X = train.drop(columns=['id', 'Churn'])
y = train['Churn'].eq('Yes').astype(int)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols_lr = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
preprocess = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_cols),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_cols_lr),
    ('binary', 'passthrough', ['SeniorCitizen']),
])
lr = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=1000))])
lr.fit(X_train, y_train)
lr_prob = lr.predict_proba(X_valid)[:, 1]
print('LR ROC-AUC:', round(roc_auc_score(y_valid, lr_prob), 4))
print('LR Average Precision:', round(average_precision_score(y_valid, lr_prob), 4))

## 4. LightGBM with native categorical features

In [ ]:
from lightgbm import LGBMClassifier

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()
cat_cols_lgb = X_train_lgb.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols_lgb:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_valid_lgb[col] = pd.Categorical(X_valid_lgb[col], categories=X_train_lgb[col].cat.categories)

lgb_model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                           random_state=42, verbosity=-1, n_jobs=-1)
lgb_model.fit(X_train_lgb, y_train, categorical_feature=cat_cols_lgb)
lgb_prob = lgb_model.predict_proba(X_valid_lgb)[:, 1]
metrics = pd.DataFrame({
    'Model': ['Logistic Regression', 'LightGBM'],
    'ROC-AUC': [roc_auc_score(y_valid, lr_prob), roc_auc_score(y_valid, lgb_prob)],
    'Average Precision': [average_precision_score(y_valid, lr_prob), average_precision_score(y_valid, lgb_prob)],
    'Brier Score': [brier_score_loss(y_valid, lr_prob), brier_score_loss(y_valid, lgb_prob)],
})
display(metrics.round(4))
print('LightGBM training ROC-AUC:', round(roc_auc_score(y_train, lgb_model.predict_proba(X_train_lgb)[:, 1]), 4))

## 5. Feature importance and calibration

In [ ]:
gain = pd.Series(lgb_model.booster_.feature_importance(importance_type='gain'), index=X_train_lgb.columns).sort_values(ascending=False)
display((gain / gain.sum() * 100).head(12).round(2).rename('Gain share (%)').to_frame())
ax = gain.head(12).sort_values().plot.barh(figsize=(8, 5), title='LightGBM feature importance (gain)')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
fraction_positive, mean_predicted = calibration_curve(y_valid, lgb_prob, n_bins=10, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], '--', label='Perfect calibration')
plt.plot(mean_predicted, fraction_positive, 'o-', label='LightGBM')
plt.xlabel('Mean predicted churn probability')
plt.ylabel('Observed churn frequency')
plt.title('Validation calibration')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Customer targeting: Top-K metrics

In [ ]:
def top_k_table(y_true, y_prob, fractions=(0.1, 0.2, 0.3)):
    labels = np.asarray(y_true)
    ranking = np.argsort(-np.asarray(y_prob), kind='stable')
    n = len(labels)
    prevalence = labels.mean()
    rows = []
    for fraction in fractions:
        k = int(n * fraction)
        tp = int(labels[ranking[:k]].sum())
        precision = tp / k
        recall = tp / labels.sum()
        rows.append({'Contact fraction': fraction, 'Contacted': k, 'True churners': tp,
                     'Precision@K': precision, 'Recall@K': recall, 'Lift@K': precision / prevalence})
    return pd.DataFrame(rows)

topk_lgb = top_k_table(y_valid, lgb_prob)
topk_lr = top_k_table(y_valid, lr_prob)
display(topk_lgb.round(4))

## 7. Risk segmentation

In [ ]:
risk = X_valid.copy()
risk['Churn'] = np.asarray(y_valid)
risk['ChurnProbability'] = lgb_prob
labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
risk['RiskLevel'] = pd.qcut(risk['ChurnProbability'].rank(method='first'), q=5, labels=labels)
summary = risk.groupby('RiskLevel', observed=True).agg(Customers=('Churn', 'size'), ChurnRate=('Churn', 'mean'), AveragePredictedRisk=('ChurnProbability', 'mean'))
display(summary.round(4))

## 8. Hypothetical retention economics

Assumptions: **CNY 10** contact cost, **CNY 200** retained-customer value, and **10%, 20%, 30%** assumed success rates. The formula is

\[E[\mathrm{Profit}(K)] = TP@K \times q \times V - K \times C.\]

**Caution:** The success rates are hypothetical and are not causal treatment-effect estimates. Actual intervention impact would require appropriate experimentation.

In [ ]:
profit = topk_lgb[['Contact fraction', 'Contacted', 'True churners']].copy()
for q in [0.1, 0.2, 0.3]:
    profit[f'Expected profit, q={q:.0%} (CNY)'] = profit['True churners'] * q * 200 - profit['Contacted'] * 10
display(profit.round(0))
comparison = topk_lgb[['Contact fraction', 'Contacted', 'True churners']].merge(
    topk_lr[['Contact fraction', 'True churners']], on='Contact fraction', suffixes=('_LGB', '_LR'))
comparison['Additional identified churners'] = comparison['True churners_LGB'] - comparison['True churners_LR']
comparison['Additional simulated profit at q=20% (CNY)'] = comparison['Additional identified churners'] * 0.2 * 200
display(comparison[['Contact fraction', 'Additional identified churners', 'Additional simulated profit at q=20% (CNY)']])

## 9. Final model and Kaggle submission

Refit the **validated** LightGBM configuration on the entire labeled training dataset. Do not use the validation model to generate the competition submission.

In [ ]:
X_full = X.copy()
X_test = test[X.columns].copy()
cat_cols_full = X_full.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols_full:
    X_full[col] = X_full[col].astype('category')
    X_test[col] = pd.Categorical(X_test[col], categories=X_full[col].cat.categories)
final_model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                            random_state=42, verbosity=-1, n_jobs=-1)
final_model.fit(X_full, y, categorical_feature=cat_cols_full)
submission = sample_submission.copy()
submission['Churn'] = final_model.predict_proba(X_test)[:, 1]
assert submission.columns.tolist() == ['id', 'Churn']
assert submission['id'].equals(test['id'])
assert submission['Churn'].notna().all() and submission['Churn'].between(0, 1).all()
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Saved /kaggle/working/submission.csv', submission.shape)
display(submission.head())

## 10. Conclusions and limitations

- The project compares an interpretable statistical baseline with a gradient-boosted tree model using a consistent stratified validation split.
- Churn risk ranking can support prioritization under limited contact budgets.
- The competition data is synthetic; no external business generalization is claimed.
- Contract-type differences and model importance are **associations, not causal effects**.
- Retention profit estimates are **simulations**, not observed campaign returns.
- The original project includes additional EDA, experiment notes, and reusable code in the [GitHub repository](https://github.com/tj-xin/customer-churn-analytics).

**Competition status:** The previously recorded Kaggle Public ROC-AUC was 0.91295 and Private ROC-AUC was 0.91415 from an after-deadline submission. These are historical submission results, not outputs of this notebook run or an official in-competition ranking.